# 02 - Hiperparámetros, Regularización y Early Stopping en XGBoost

## Curso de Machine Learning: Modelos de Regresión

En este cuaderno profundizaremos en el ajuste sistemático de hiperparámetros en `xgboost.XGBRegressor`, dominaremos la técnica de **Parada Temprana (Early Stopping)** para detener el entrenamiento en el punto óptimo de generalización y compararemos el desempeño de XGBoost frente a las implementaciones de Boosting de Scikit-Learn.

---

### Contenidos
1. Catálogo de Hiperparámetros Fundamentales en XGBoost
2. Implementación de Early Stopping con `early_stopping_rounds`
3. Optimización con `GridSearchCV`
4. Comparativa de Rendimiento: `GradientBoostingRegressor`, `HistGradientBoostingRegressor` y `XGBRegressor`
5. Conclusiones


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## 1. Carga de Datos Auto MPG

In [ ]:
df = pd.read_csv("../../data/auto_mpg.csv")
features = ['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year']
X = df[features]
y = df['mpg']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)


## 2. Parada Temprana (Early Stopping)

A diferencia de Random Forest (donde agregar árboles nunca causa sobreajuste), en Boosting **un exceso de árboles sí provocará sobreajuste** porque el modelo terminará memorizando el ruido residual.

Con `early_stopping_rounds=K`, el entrenamiento se interrumpe automáticamente si la métrica de validación no mejora durante $K$ iteraciones consecutivas, conservando el mejor modelo encontrado:


In [ ]:
xgb_early = xgb.XGBRegressor(
    n_estimators=500, # Valor deliberadamente alto
    learning_rate=0.05,
    max_depth=3,
    early_stopping_rounds=15,
    random_state=42,
    eval_metric='rmse'
)

xgb_early.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

print(f"Mejor iteracion encontrada: {xgb_early.best_iteration}")
print(f"Mejor puntuacion RMSE en validacion: {xgb_early.best_score:.4f} mpg")

y_pred_e = xgb_early.predict(X_test)
print(f"R2 en Prueba con Early Stopping: {r2_score(y_test, y_pred_e):.4f}")
print(f"RMSE en Prueba:                  {np.sqrt(mean_squared_error(y_test, y_pred_e)):.2f} mpg")


## 3. Búsqueda por Rejilla (`GridSearchCV`)

Calibramos los hiperparámetros estructurales de los árboles (`max_depth`), el submuestreo de observaciones (`subsample`) y el submuestreo de características (`colsample_bytree`):


In [ ]:
param_grid = {
    'n_estimators': [80, 120],
    'learning_rate': [0.03, 0.08],
    'max_depth': [3, 4, 5],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

grid_xgb = GridSearchCV(
    xgb.XGBRegressor(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)
grid_xgb.fit(X_train, y_train)

print(f"Mejores parametros: {grid_xgb.best_params_}")
print(f"Mejor R2 CV:        {grid_xgb.best_score_:.4f}")

best_xgb = grid_xgb.best_estimator_
y_pred_best = best_xgb.predict(X_test)
print(f"R2 en Test:         {r2_score(y_test, y_pred_best):.4f}")


## 4. Benchmark Comparativo de Algoritmos de Boosting

Comparamos tres enfoques de Boosting:
1. `GradientBoostingRegressor` (Árboles exactos tradicionales en Scikit-Learn)
2. `HistGradientBoostingRegressor` (Implementación rápida basada en histogramas de Scikit-Learn)
3. `XGBRegressor` (XGBoost)


In [ ]:
modelos_boosting = {
    'Gradient Boosting (sklearn)': GradientBoostingRegressor(n_estimators=120, learning_rate=0.08, max_depth=3, random_state=42),
    'HistGradientBoosting (sklearn)': HistGradientBoostingRegressor(max_iter=120, learning_rate=0.08, max_depth=3, random_state=42),
    'XGBoost': xgb.XGBRegressor(n_estimators=120, learning_rate=0.08, max_depth=3, random_state=42)
}

bench_res = []

for nom, mod in modelos_boosting.items():
    t0 = time.perf_counter()
    mod.fit(X_train, y_train)
    t_train = (time.perf_counter() - t0) * 1000
    
    t0 = time.perf_counter()
    preds = mod.predict(X_test)
    t_pred = (time.perf_counter() - t0) * 1000
    
    bench_res.append({
        'Modelo': nom,
        'R2 Test': r2_score(y_test, preds),
        'RMSE Test': np.sqrt(mean_squared_error(y_test, preds)),
        'Tiempo Train (ms)': t_train,
        'Tiempo Pred (ms)': t_pred
    })

df_bench = pd.DataFrame(bench_res)
print(df_bench.to_string(index=False))


## 5. Conclusiones

1. **Early Stopping**: Es una práctica indispensable en Boosting para evitar el sobreajuste y ahorrar tiempo de cómputo.
2. **Submuestreo**: Combinar `subsample < 1.0` y `colsample_bytree < 1.0` añade aleatoriedad que incrementa la robustez del modelo.
3. **Escalabilidad**: Tanto XGBoost como HistGradientBoosting son sustancialmente más rápidos en datasets de gran volumen.
